<h3>Pipeline Creation for K-Folds Training

In [74]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from evaluate_binary_classifier import *

import joblib

from scipy.stats import randint, uniform, loguniform

from sklearn.preprocessing import RobustScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
    GridSearchCV
)
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

PROJ_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJ_ROOT / "data"
INTERIM_DATA_DIR = DATA_DIR / "interim"
RAW_DATA_DIR = DATA_DIR / "raw"

In [49]:
with open(INTERIM_DATA_DIR / "X_test_raw.pkl", "rb") as f:
    X_test = pickle.load(f)

with open(INTERIM_DATA_DIR / "y_test_raw.pkl", "rb") as f:
    y_test = pickle.load(f)

with open(INTERIM_DATA_DIR / "X_train_raw.pkl", "rb") as f:
    X_train = pickle.load(f)

with open(INTERIM_DATA_DIR / "y_train_raw.pkl", "rb") as f:
    y_train = pickle.load(f)

In [50]:
yes_no_map = {"no": 0, "yes": 1}

y_train = y_train.map(yes_no_map).astype("int8")
y_test = y_test.map(yes_no_map).astype("int8")

Pipeline which does all the imputation, preprocessing and sampling on the post split raw data:

In [51]:
# As we don't need to use grouped mode imputation anymore as 'contact' isn't needed, we only need to impute the 'job' / 'education' rows...

def filter_unknown_rows(
    X,
    y,
    columns=("job", "education"),
    unknown_value="unknown"
):
    """
    Remove rows where any selected column contains:
    - NaN / missing value
    - an empty string
    - whitespace only
    - the value 'unknown'

    The same rows are removed from X and y.
    """

    X = X.copy()
    y = y.copy()

    if len(X) != len(y):
        raise ValueError(
            f"X and y have different lengths: {len(X)} and {len(y)}."
        )

    if not X.index.equals(y.index):
        raise ValueError(
            "X and y indexes are not aligned."
        )

    missing_columns = [
        column
        for column in columns
        if column not in X.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Columns not found in X: {missing_columns}"
        )

    keep_mask = pd.Series(
        True,
        index=X.index
    )

    for column in columns:
        cleaned_column = (
            X[column]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        keep_mask &= cleaned_column.notna()
        keep_mask &= cleaned_column.ne("")
        keep_mask &= cleaned_column.ne(
            unknown_value.lower()
        )

    X_filtered = X.loc[keep_mask].copy()
    y_filtered = y.loc[keep_mask].copy()

    return X_filtered, y_filtered

In [52]:
# Apply independently to train and test
X_train, y_train = filter_unknown_rows(
    X_train,
    y_train,
    columns=("job", "education")
)

X_test, y_test = filter_unknown_rows(
    X_test,
    y_test,
    columns=("job", "education")
)

# Reset indexes after deleting rows
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("\nFiltered training shape:", X_train.shape)
print("Filtered test shape:", X_test.shape)

print(
    "Training X/y aligned:",
    len(X_train) == len(y_train)
)

print(
    "Test X/y aligned:",
    len(X_test) == len(y_test)
)


Filtered training shape: (26811, 13)
Filtered test shape: (11527, 13)
Training X/y aligned: True
Test X/y aligned: True


In [53]:
# Feature dropping (custom transformer adapted from the previous notebooks)

class FeatureDropper(BaseEstimator, TransformerMixin):
    """
    Delete selected dataframe columns.
    """

    def __init__(self, columns=None):
        self.columns = columns

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError(
                "FeatureDropper requires a pandas DataFrame."
            )

        self.columns_ = list(self.columns or [])

        missing_columns = [
            column
            for column in self.columns_
            if column not in X.columns
        ]

        if missing_columns:
            raise KeyError(
                f"Columns not found: {missing_columns}"
            )

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError(
                "FeatureDropper requires a pandas DataFrame."
            )

        return X.drop(
            columns=self.columns_
        ).copy()

In [54]:
# Features to apply to:

features_to_remove = [
    "age",
    "balance",
    "campaign",
    "default",
    "loan",
    "contact"
]

In [55]:
# Cyclical encoding of day and month columns:

class CyclicalDateEncoder(BaseEstimator, TransformerMixin):
    """
    Convert day and month into sine/cosine cyclical features.

    Creates:
    - day_sin
    - day_cos
    - month_sin
    - month_cos

    The original day and month columns are then removed.
    """

    def __init__(
        self,
        day_column="day",
        month_column="month",
        drop_original=True
    ):
        self.day_column = day_column
        self.month_column = month_column
        self.drop_original = drop_original

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError(
                "CyclicalDateEncoder requires a pandas DataFrame."
            )

        required_columns = [
            self.day_column,
            self.month_column
        ]

        missing_columns = [
            column
            for column in required_columns
            if column not in X.columns
        ]

        if missing_columns:
            raise KeyError(
                f"Missing cyclical columns: {missing_columns}"
            )

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError(
                "CyclicalDateEncoder requires a pandas DataFrame."
            )

        X = X.copy()

        month_mapping = {
            "jan": 1,
            "feb": 2,
            "mar": 3,
            "apr": 4,
            "may": 5,
            "jun": 6,
            "jul": 7,
            "aug": 8,
            "sep": 9,
            "oct": 10,
            "nov": 11,
            "dec": 12
        }

        day_numeric = pd.to_numeric(
            X[self.day_column],
            errors="coerce"
        )

        if pd.api.types.is_numeric_dtype(
            X[self.month_column]
        ):
            month_numeric = pd.to_numeric(
                X[self.month_column],
                errors="coerce"
            )
        else:
            month_numeric = (
                X[self.month_column]
                .astype("string")
                .str.strip()
                .str.lower()
                .map(month_mapping)
            )

        # Day cycle: 1 to 31
        X["day_sin"] = np.sin(
            2 * np.pi * (day_numeric - 1) / 31
        )

        X["day_cos"] = np.cos(
            2 * np.pi * (day_numeric - 1) / 31
        )

        # Month cycle: 1 to 12
        X["month_sin"] = np.sin(
            2 * np.pi * (month_numeric - 1) / 12
        )

        X["month_cos"] = np.cos(
            2 * np.pi * (month_numeric - 1) / 12
        )

        if self.drop_original:
            X = X.drop(
                columns=[
                    self.day_column,
                    self.month_column
                ]
            )

        return X

In [56]:
# Define all the features we are working with after these steps:

numeric_features = [
    "duration",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos"
]

housing_features = [
    "housing"
]

one_hot_features = [
    "job",
    "marital",
    "education",
]

In [57]:
X_train.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign
0,41,blue-collar,married,secondary,no,0,yes,no,cellular,1,apr,625,2
1,42,management,single,tertiary,no,-56,yes,no,unknown,20,jun,126,14
2,32,technician,married,tertiary,no,240,yes,yes,cellular,14,jul,86,3
3,24,blue-collar,single,secondary,no,-287,yes,no,cellular,18,may,504,1
4,54,blue-collar,married,secondary,no,1070,no,no,cellular,18,nov,373,4


In [58]:
# Numeric pipeline:

numeric_pipeline = SklearnPipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "robust_scaler",
            RobustScaler()
        )
    ]
)

In [59]:
# Ordinal encoding (housing) pipeline:

housing_pipeline = SklearnPipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "ordinal_encoder",
            OrdinalEncoder(
                categories=[["no", "yes"]],
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ]
)

In [60]:
# One-hot encoding:

one_hot_pipeline = SklearnPipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "one_hot_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

In [61]:
# Column transformer (define which pipelines to apply to which columns):


column_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "housing",
            housing_pipeline,
            housing_features
        ),
        (
            "categorical",
            one_hot_pipeline,
            one_hot_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


In [62]:
# Define the model:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=13,
    n_jobs=1
)

In [71]:
# Define the overall model pipeline:

xgb_model_pipeline = ImbPipeline(
    steps=[
        (
            "feature_selection",
            FeatureDropper(
                columns=features_to_remove
            )
        ),
        (
            "cyclical_encoding",
            CyclicalDateEncoder(
                day_column="day",
                month_column="month",
                drop_original=True
            )
        ),
        (
            "preprocessor",
            column_preprocessor # All encoding + scaling (apart from cyclical encoding which is done above)
        ),
        (
            "sampler",
            SMOTE(
                sampling_strategy="auto",
                k_neighbors=15,
                random_state=13
            )
        ),
        (
            "model", # This is the part to change between trying different models but with the same processing pipeline.
            xgb_model
        )
    ]
)

In [72]:
# Application:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=13
)

param_distributions = {
    "sampler__sampling_strategy": [
        0.25,
        0.40,
        0.60,
        0.80
    ],
    "model__n_estimators": randint(
        100,
        1000
    ),
    "model__learning_rate": loguniform(
        0.1,
        0.12
    ),
    "model__max_depth": randint(
        2,
        6
    ),
    "model__min_child_weight": randint(
        3,
        21
    ),
    "model__gamma": uniform(
        0,
        1.5
    ),
    "model__subsample": uniform(
        0.60,
        0.35
    ),
    "model__colsample_bytree": uniform(
        0.60,
        0.35
    ),
    "model__reg_alpha": loguniform(
        1e-4,
        2
    ),

    "model__reg_lambda": loguniform(
        1,
        30
    )
}

search_xgb = RandomizedSearchCV(
    estimator=xgb_model_pipeline,
    param_distributions=param_distributions,
    n_iter=180,
    scoring="f1",
    cv=cv,
    refit=True,
    return_train_score=True,
    random_state=13,
    n_jobs=-1,
    verbose=2,
    error_score="raise"
)

search_xgb.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 180 candidates, totalling 900 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__colsample_bytree': <scipy.stats....0023092A87050>, 'model__gamma': <scipy.stats....0023092A870B0>, 'model__learning_rate': <scipy.stats....0023092B1E1E0>, 'model__max_depth': <scipy.stats....0023091A09D90>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",180
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Ref

In [70]:
# Training score
print("Best 5-fold CV F1-score:", search_xgb.best_score_)
print("Best parameters:")
print(search_xgb.best_params_)

# Test score:
best_xgb = search_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:,1]

best_xgb_evaluate = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)

Best 5-fold CV F1-score: 0.5537448932282288
Best parameters:
{'model__colsample_bytree': np.float64(0.7930668191742306), 'model__gamma': np.float64(0.32959003815812526), 'model__learning_rate': np.float64(0.1054080539897115), 'model__max_depth': 5, 'model__min_child_weight': 15, 'model__n_estimators': 149, 'model__reg_alpha': np.float64(1.3786202225377886), 'model__reg_lambda': np.float64(27.07004050085197), 'model__subsample': np.float64(0.7553483454376724), 'sampler__sampling_strategy': 0.8}

===== Binary Classification Evaluation =====

Accuracy: 0.9214
Balanced Accuracy: 0.7929
Precision: 0.4721
Recall: 0.6425
F1 Score: 0.5443
Matthews Corrcoef: 0.5095
ROC AUC: 0.9378
Log Loss: 0.1680

Confusion Matrix:
[[10080   605]
 [  301   541]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.94      0.96     10685
           1       0.47      0.64      0.54       842

    accuracy                           0.92     11527
   macro a

Both the training and test F1-scores are around 55%. This seems more reasonable, and suggests that there isn't overfitting occuring. Save this pipeline as its own xgb specific pipeline and then test some other model types.

<h4>Logistic Regression Pipeline

In [78]:
from sklearn.linear_model import LogisticRegression


lr_model = LogisticRegression(max_iter=1000, random_state=13)

lr_model_pipeline = ImbPipeline(
    steps=[
        (
            "feature_selection",
            FeatureDropper(
                columns=features_to_remove
            )
        ),
        (
            "cyclical_encoding",
            CyclicalDateEncoder(
                day_column="day",
                month_column="month",
                drop_original=True
            )
        ),
        (
            "preprocessor",
            column_preprocessor # All encoding + scaling (apart from cyclical encoding which is done above)
        ),
        (
            "sampler",
            SMOTE(
                sampling_strategy="auto",
                k_neighbors=15,
                random_state=13
            )
        ),
        (
            "model",
            lr_model
        )
    ]
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=13
)

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"],
    "model__class_weight": [None, "balanced"]
}

search_lr = GridSearchCV(
    estimator=lr_model_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
    verbose=2,
    error_score="raise"
)

search_lr.fit(
    X_train,
    y_train
)

# Training score
print("Best 5-fold CV F1-score:", search_lr.best_score_)
print("Best parameters:")
print(search_lr.best_params_)

# Test score:
best_lr = search_lr.best_estimator_
y_pred = best_lr.predict(X_test)
y_proba = best_lr.predict_proba(X_test)[:,1]

best_lr_evaluate = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best 5-fold CV F1-score: 0.44034143159511097
Best parameters:
{'model__C': 0.01, 'model__class_weight': None, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}

===== Binary Classification Evaluation =====

Accuracy: 0.8479
Balanced Accuracy: 0.8195
Precision: 0.2962
Recall: 0.7862
F1 Score: 0.4303
Matthews Corrcoef: 0.4206
ROC AUC: 0.9002
Log Loss: 0.4383

Confusion Matrix:
[[9112 1573]
 [ 180  662]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.85      0.91     10685
           1       0.30      0.79      0.43       842

    accuracy                           0.85     11527
   macro avg       0.64      0.82      0.67     11527
weighted avg       0.93      0.85      0.88     11527



c:\Users\cochr\Data Science Projects\lA82lBNDpo9AlAnK\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


<h4>KNN Pipeline